# Graph RAG
### Text-to-Cypher retrieval over a live Neo4j customer graph

Same customer/product/order data as `SQL_RAG.ipynb`, modeled as a graph instead of tables: `Customer` and `Product` nodes, `PURCHASED` and `REFERRED` relationships.

Retrieval here means: turn the question into **Cypher**, run it against **Neo4j**, and answer from the returned nodes/relationships — the same text-to-query idea as Notebook 1, just for a different data shape.

**Before running this notebook:**
```bash
docker compose up -d
```
(run from this folder — brings up Neo4j on `localhost:7687`, browser UI at `http://localhost:7474`)

## Step 1: Connect to the database

In [1]:
!pip install langchain langchain-community langchain-openai langchain-neo4j neo4j python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI

graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="neo4j_password",
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Connected:", graph.query("RETURN 1 AS ok"))

C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected: [{'ok': 1}]


## Step 2: Seed the graph
The same 20 customers and 23 orders from `SQL_RAG.ipynb`, reshaped: each customer's `referred_by_id` becomes a `REFERRED` edge (referrer -> referred), each order becomes a `PURCHASED` edge (customer -> product). `MERGE` throughout so re-running this cell is safe.

In [3]:
CUSTOMERS = [
    (1,  "Aria Kowalski",     "Berlin",     "Germany", "Pro",        None),
    (2,  "Noah Fischer",      "Munich",     "Germany", "Basic",      1),
    (3,  "Mia Andersson",     "Stockholm",  "Sweden",  "Enterprise", None),
    (4,  "Liam O'Brien",      "Dublin",     "Ireland", "Pro",        3),
    (5,  "Sofia Rossi",       "Milan",      "Italy",   "Basic",      None),
    (6,  "Lucas Dubois",      "Lyon",       "France",  "Pro",        5),
    (7,  "Emma Johansson",    "Gothenburg", "Sweden",  "Basic",      3),
    (8,  "Ethan Murphy",      "Cork",       "Ireland", "Enterprise", 4),
    (9,  "Olivia Bianchi",    "Rome",       "Italy",   "Pro",        5),
    (10, "William Nowak",     "Warsaw",     "Poland",  "Basic",      None),
    (11, "Ava Kowalczyk",     "Krakow",     "Poland",  "Pro",        10),
    (12, "James Laurent",     "Paris",      "France",  "Enterprise", 6),
    (13, "Isabella Conti",    "Turin",      "Italy",   "Basic",      9),
    (14, "Benjamin Novak",    "Brno",       "Czechia", "Pro",        None),
    (15, "Charlotte Larsen",  "Aarhus",     "Denmark", "Basic",      3),
    (16, "Henrik Svensson",   "Malmo",      "Sweden",  "Pro",        7),
    (17, "Amelia Walsh",      "Galway",     "Ireland", "Enterprise", 8),
    (18, "Daniel Kaczmarek",  "Gdansk",     "Poland",  "Basic",      10),
    (19, "Grace Moreau",      "Nice",       "France",  "Pro",        6),
    (20, "Oscar Bergstrom",   "Uppsala",    "Sweden",  "Basic",      16),
]

# (customer_id, product, amount)
ORDERS = [
    (1,  "Cloud Suite",   199.00), (2,  "Support Plus",   79.00), (3,  "AI Toolkit",     499.00),
    (3,  "Analytics Pro", 349.00), (4,  "Cloud Suite",   199.00), (5,  "DataSync",       129.00),
    (6,  "Analytics Pro", 349.00), (7,  "Support Plus",   79.00), (8,  "AI Toolkit",     499.00),
    (9,  "DataSync",      129.00), (10, "AI Toolkit",    499.00), (11, "Cloud Suite",    199.00),
    (12, "Analytics Pro", 349.00), (13, "Support Plus",   79.00), (14, "DataSync",       129.00),
    (15, "Cloud Suite",   199.00), (16, "AI Toolkit",    499.00), (17, "Analytics Pro",  349.00),
    (18, "Support Plus",  79.00),  (19, "DataSync",      129.00), (20, "Cloud Suite",    199.00),
    (1,  "AI Toolkit",    499.00), (10, "Cloud Suite",   199.00),
]

graph.query("MATCH (n) DETACH DELETE n")  # start clean

for cid, name, city, country, plan, _ in CUSTOMERS:
    graph.query(
        "MERGE (c:Customer {customer_id: $cid}) SET c.name=$name, c.city=$city, c.country=$country, c.plan=$plan",
        {"cid": cid, "name": name, "city": city, "country": country, "plan": plan},
    )

for cid, name, city, country, plan, ref in CUSTOMERS:
    if ref is not None:
        graph.query(
            """MATCH (referrer:Customer {customer_id: $ref}), (referred:Customer {customer_id: $cid})
               MERGE (referrer)-[:REFERRED]->(referred)""",
            {"ref": ref, "cid": cid},
        )

for cid, product, amount in ORDERS:
    graph.query(
        """MERGE (p:Product {name: $product})
           WITH p
           MATCH (c:Customer {customer_id: $cid})
           MERGE (c)-[:PURCHASED {amount: $amount}]->(p)""",
        {"cid": cid, "product": product, "amount": amount},
    )

counts = graph.query(
    "MATCH (c:Customer) WITH count(c) AS customers "
    "MATCH (p:Product) WITH customers, count(p) AS products "
    "MATCH ()-[r:PURCHASED]->() WITH customers, products, count(r) AS purchases "
    "MATCH ()-[f:REFERRED]->() RETURN customers, products, purchases, count(f) AS referrals"
)
print(counts[0])

{'customers': 20, 'products': 5, 'purchases': 23, 'referrals': 15}


## Step 3: Inspect the schema
`refresh_schema()` walks the live graph and produces the node labels, relationship types, and properties the model needs — same role `SQLDatabase.get_table_info()` played in Notebook 1.

In [4]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Customer {plan: STRING, city: STRING, country: STRING, customer_id: INTEGER, name: STRING}
Product {name: STRING}
Relationship properties:
PURCHASED {amount: FLOAT}
The relationships:
(:Customer)-[:PURCHASED]->(:Product)
(:Customer)-[:REFERRED]->(:Customer)


## Step 4: Text-to-Cypher
Same shape as `generate_sql` in Notebook 1 — schema in, one query out — just a different query language.

In [5]:
CYPHER_PROMPT = """You are a Neo4j Cypher expert. Given the schema below, write ONE read-only Cypher query
that answers the question. Output ONLY the Cypher, no explanation, no markdown fences.

If the question chains two conditions about two different people (e.g. "customers referred by
someone who bought X"), do NOT force both relationships into a single continuous path -- write
each condition as its own MATCH clause sharing a common variable for the node both conditions
refer to. For example, "customers referred by someone who bought X" is two separate facts about
the referrer, so:
MATCH (referrer:Customer)-[:PURCHASED]->(:Product {{name: 'X'}})
MATCH (referrer)-[:REFERRED]->(referred:Customer)
RETURN DISTINCT referred.name
Only chain relationships into a single path when the question describes one continuous hop
(A connects to B connects to C).

Schema:
{schema}

Question: {question}
Cypher:"""

def generate_cypher(question, schema):
    cypher = llm.invoke(CYPHER_PROMPT.format(schema=schema, question=question)).content.strip()
    return cypher.strip("`").removeprefix("cypher").strip() if cypher.lower().startswith("```") else cypher

## Step 5: Safety guardrail
Same read-only discipline as Notebook 1's `is_safe_select` — block any write clause before it ever reaches the graph.

In [6]:
import re

WRITE_KEYWORDS = re.compile(r"\b(CREATE|MERGE|DELETE|SET|REMOVE|DROP)\b", re.IGNORECASE)

def is_read_only(cypher):
    return not WRITE_KEYWORDS.search(cypher)

## Step 6: Execute + synthesize the answer
No retry loop this time (kept to Notebook 1, to avoid repeating the same mechanism twice) — straight execute-then-answer.

In [7]:
ANSWER_PROMPT = """Answer the question using only the query result rows below.

Question: {question}
Rows: {rows}

Answer:"""

def graph_rag(question):
    cypher = generate_cypher(question, graph.schema)
    if not is_read_only(cypher):
        raise ValueError(f"Refused to execute a write query: {cypher}")
    rows = graph.query(cypher)
    answer = llm.invoke(ANSWER_PROMPT.format(question=question, rows=rows)).content.strip()
    return cypher, rows, answer

## Step 7: Try it — one hop, then two
The last question needs two hops (referral, then purchase) chained together — the kind of query that turns into a self-join in SQL but reads as a straight-line path here.

In [8]:
test_questions = [
    "Which customers did Sofia Rossi refer?",
    "Which customers were referred by someone who purchased the AI Toolkit?",
    "Which products were purchased by people referred by Mia Andersson?",
]

for q in test_questions:
    cypher, rows, answer = graph_rag(q)
    print(f"--- {q} ---")
    print(f"Cypher: {cypher}")
    print(f"Rows returned: {len(rows)}")
    print(f"Answer: {answer}\n")

--- Which customers did Sofia Rossi refer? ---
Cypher: MATCH (referrer:Customer {name: 'Sofia Rossi'})-[:REFERRED]->(referred:Customer)
RETURN DISTINCT referred.name
Rows returned: 2
Answer: Sofia Rossi referred Olivia Bianchi and Lucas Dubois.



--- Which customers were referred by someone who purchased the AI Toolkit? ---
Cypher: MATCH (referrer:Customer)-[:PURCHASED]->(:Product {name: 'AI Toolkit'})
MATCH (referrer)-[:REFERRED]->(referred:Customer)
RETURN DISTINCT referred.name
Rows returned: 8
Answer: Noah Fischer, Oscar Bergstrom, Daniel Kaczmarek, Ava Kowalczyk, Amelia Walsh, Charlotte Larsen, Emma Johansson, Liam O'Brien.



--- Which products were purchased by people referred by Mia Andersson? ---
Cypher: MATCH (referrer:Customer {name: 'Mia Andersson'})-[:REFERRED]->(referred:Customer)
MATCH (referred)-[:PURCHASED]->(product:Product)
RETURN DISTINCT product.name
Rows returned: 2
Answer: The products purchased by people referred by Mia Andersson are Cloud Suite and Support Plus.



## Step 8: The guardrail in action

In [9]:
dangerous_question = "Delete the customer named William Nowak."
cypher = generate_cypher(dangerous_question, graph.schema)
print(f"Generated: {cypher}")
print(f"Read-only: {is_read_only(cypher)}")

try:
    graph_rag(dangerous_question)
except ValueError as e:
    print(f"Refused: {e}")

Generated: MATCH (customer:Customer {name: 'William Nowak'})
DELETE customer
Read-only: False


Refused: Refused to execute a write query: MATCH (customer:Customer {name: 'William Nowak'})
DELETE customer


## Try it yourself
1. Ask the three-hop version: "Which products were purchased by people referred by someone in Sweden?" and check the generated Cypher actually chains `REFERRED` twice.
2. Swap `generate_cypher`'s hand-written prompt for LangChain's built-in `GraphCypherQAChain` (`from langchain_neo4j import GraphCypherQAChain`) and compare the Cypher it produces for the same questions.
3. Re-run Notebook 1's referral-style question ("which customers were referred by a customer in Poland?") here instead, and compare how much simpler the Cypher is versus the SQL self-join it would take there.

## Cleanup
```bash
docker compose down -v
```
(run from this folder — stops both containers and deletes their data volumes)